In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error
import os

# Путь к погодному датасету
weather_path = "../data/weather/RusWeather-GF_1980-2023.csv"

# Читаем (файл большой, можно сначала nrows=10000 для проверки)
df = pd.read_csv(weather_path)

# Посмотри колонки и подставь свои
print(df.columns.tolist())
print(df.head())

# !!! Укажи здесь названия своих колонок !!!
date_col = "date"          # например, 'date' или 'DATE'
target_col = "temperature" # например, 'temperature' или 'T'

df[date_col] = pd.to_datetime(df[date_col])
df = df.sort_values(date_col).set_index(date_col)

# Агрегируем по дням (если данные реже/чаще)
df_daily = df[[target_col]].resample("D").mean().dropna()

# Лаги
for lag in [1, 2, 3, 7, 14, 30]:
    df_daily[f"lag_{lag}"] = df_daily[target_col].shift(lag)

df_daily["rolling_mean_7"] = df_daily[target_col].shift(1).rolling(7).mean()
df_daily["rolling_mean_30"] = df_daily[target_col].shift(1).rolling(30).mean()
df_daily["day_of_year"] = df_daily.index.dayofyear
df_daily["month"] = df_daily.index.month

df_model = df_daily.dropna().copy()

features = [c for c in df_model.columns if c != target_col]
target = target_col

# Разделение: последний год — тест
split_date = df_model.index.max() - pd.DateOffset(years=1)
train = df_model[df_model.index < split_date]
test = df_model[df_model.index >= split_date]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

# Модель
model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

# Оценка
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f"MAE: {mae:.2f}, RMSE: {rmse:.2f}")

# Рекурсивный прогноз на 30 дней
history = df_model[[target_col]].copy()
last_date = df_model.index.max()
forecast_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=30, freq="D")
forecasts = []

for date in forecast_dates:
    row = {}
    for lag in [1, 2, 3, 7, 14, 30]:
        row[f"lag_{lag}"] = history[target_col].iloc[-lag]
    row["rolling_mean_7"] = history[target_col].iloc[-7:].mean()
    row["rolling_mean_30"] = history[target_col].iloc[-30:].mean()
    row["day_of_year"] = date.dayofyear
    row["month"] = date.month
    X_pred = pd.DataFrame([row])[features]
    pred = model.predict(X_pred)[0]
    forecasts.append({"date": date, target_col: pred})
    history = pd.concat([
        history,
        pd.DataFrame({target_col: [pred]}, index=[date])
    ])

forecast_df = pd.DataFrame(forecasts)
print(forecast_df.head())

# Сохранение
os.makedirs("../models", exist_ok=True)
joblib.dump({
    "model": model,
    "features": features,
    "date_col": date_col,
    "target_col": target_col
}, "../models/weather_lgbm.joblib")
print("Модель сохранена: ../models/weather_lgbm.joblib")

['station_id', 'date', 'year', 'month', 'day', 'temperature', 'precipitation', 'lat', 'lon', 'elevation']
   station_id        date  year  month  day  temperature  precipitation   lat  \
0       20046  1980-01-01  1980      1    1        -24.5            1.7  80.6   
1       20046  1980-01-02  1980      1    2        -21.8            0.5  80.6   
2       20046  1980-01-03  1980      1    3        -24.3            0.5  80.6   
3       20046  1980-01-04  1980      1    4        -24.0            5.2  80.6   
4       20046  1980-01-05  1980      1    5        -18.1            3.7  80.6   

    lon  elevation  
0  58.0       21.0  
1  58.0       21.0  
2  58.0       21.0  
3  58.0       21.0  
4  58.0       21.0  
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005014 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2308
[LightGBM] [Info] Number of data points in the train set: 15675, number of used featu